# 10 学习率 Warmup、稳定段和衰减应怎样设计？

## 面试回答主线

学习率调度要回答三个问题：初期何时从小步升到峰值、峰值保持多久以高效学习、何时衰减以收敛。Warmup 的作用是让随机初始化、归一化统计和优化器状态进入可控区，而不是机械地设置固定百分比。稳定段与衰减段则应按 token、batch 配方、验证曲线和恢复策略共同决定。实验手写线性 warmup + 常量平台 + cosine decay，并在同一客服分类任务上比较“首步峰值且更新符号损坏”的事故与健康 schedule。它演示调度状态机，不是生产学习率搜索。

**核心公式：** 线性 warmup 可写为 $\eta_t=\eta_{max}t/T_w$；平台段取 $\eta_{max}$；cosine 衰减为 $\eta_t=\eta_{min}+\frac12(\eta_{max}-\eta_{min})(1+\cos(\pi u))$。

本 Notebook 将依次展示业务输入、可比较基线、手写核心机制、中间量、失败与修复；所有数值都是确定性的教学实验。


## 真实案例

场景是支付与账户安全客服系统：模型要把工单分成“高风险需优先人工处理”和“常规处理”。三个输入特征分别表示资金风险线索、登录/身份线索和售后/账单线索。数据为人工构造的脱敏离线事件，字段结构模拟真实工单，不可外推为生产表现。


In [1]:
import math  # 导入数学函数以实现尺度公式。
import warnings  # 导入警告控制模块以保持教学输出干净。
warnings.filterwarnings('ignore', message='The pynvml package is deprecated')  # 屏蔽环境依赖产生的非教学弃用警告。
import torch  # 导入 PyTorch 张量和自动微分能力。
import torch.nn as nn  # 导入模块基类以手写网络结构。
torch.manual_seed(17)  # 固定随机种子使教学输出可复现。
torch.set_num_threads(1)  # 限制 CPU 线程以减少小实验波动。
samples = [  # 构造脱敏客服工单的真实语义样本。
    {'ticket': '支付重复扣款，要求退款', 'features': [1.0, 0.0, 1.0], 'label': 1},  # 高风险退款工单。
    {'ticket': '登录验证码收不到', 'features': [0.0, 1.0, 0.0], 'label': 0},  # 普通技术支持工单。
    {'ticket': '账户出现陌生转账', 'features': [1.0, 0.0, 0.0], 'label': 1},  # 高风险资金安全工单。
    {'ticket': '如何修改收货地址', 'features': [0.0, 0.0, 1.0], 'label': 0},  # 普通售后咨询工单。
    {'ticket': '银行卡被盗刷请冻结', 'features': [1.0, 1.0, 0.0], 'label': 1},  # 高风险且紧急的工单。
    {'ticket': '发票抬头需要更正', 'features': [0.0, 1.0, 1.0], 'label': 0},  # 低风险但需要人工处理的工单。
]  # 结束教学样本定义。
features = torch.tensor([row['features'] for row in samples], dtype=torch.float32)  # 将可读字段转为模型输入张量。
labels = torch.tensor([row['label'] for row in samples], dtype=torch.long)  # 将风险标签转为分类目标。
print('教学实验：脱敏客服工单，不代表线上规模或泛化收益。')  # 明确实验边界。
for row in samples:  # 逐条展示输入样本而不是隐藏在张量中。
    print(f"标签={row['label']} | 特征={row['features']} | 工单={row['ticket']}")  # 输出原始业务语义。
print(f'输入张量形状={tuple(features.shape)}，标签={labels.tolist()}')  # 输出张量形状和目标。


教学实验：脱敏客服工单，不代表线上规模或泛化收益。
标签=1 | 特征=[1.0, 0.0, 1.0] | 工单=支付重复扣款，要求退款
标签=0 | 特征=[0.0, 1.0, 0.0] | 工单=登录验证码收不到
标签=1 | 特征=[1.0, 0.0, 0.0] | 工单=账户出现陌生转账
标签=0 | 特征=[0.0, 0.0, 1.0] | 工单=如何修改收货地址
标签=1 | 特征=[1.0, 1.0, 0.0] | 工单=银行卡被盗刷请冻结
标签=0 | 特征=[0.0, 1.0, 1.0] | 工单=发票抬头需要更正
输入张量形状=(6, 3)，标签=[1, 0, 1, 0, 1, 0]


## Baseline / 基线

先看最简单的对照。基线与核心方案使用完全相同的样本、标签和指标，避免把数据变化误认为算法收益。


In [2]:
def make_schedule(total_steps, warmup_steps, hold_steps, peak, floor):  # 手写 warmup、平台和 cosine decay 调度器。
    values = []  # 保存每步学习率。
    for step in range(total_steps):  # 逐步构造调度值。
        if step < warmup_steps:  # 处理线性 warmup 区间。
            value = peak * float(step + 1) / float(warmup_steps)  # 从小学习率线性升到峰值。
        elif step < warmup_steps + hold_steps:  # 处理稳定平台区间。
            value = peak  # 保持峰值学习率。
        else:  # 处理余下的 cosine 衰减区间。
            progress = float(step - warmup_steps - hold_steps + 1) / float(total_steps - warmup_steps - hold_steps)  # 计算衰减进度。
            value = floor + 0.5 * (peak - floor) * (1.0 + math.cos(math.pi * progress))  # 计算 cosine 学习率。
        values.append(value)  # 保存当前学习率。
    return values  # 返回完整可复现 schedule。
healthy_schedule = make_schedule(18, 4, 6, 0.55, 0.05)  # 创建健康的三段式学习率表。
baseline_metric = healthy_schedule[0]  # 记录 warmup 首步学习率。
print(f'健康 schedule 前 6 步={ [round(value, 3) for value in healthy_schedule[:6]] }，后 4 步={ [round(value, 3) for value in healthy_schedule[-4:]] }')  # 展示调度状态机输出。


健康 schedule 前 6 步=[0.138, 0.275, 0.413, 0.55, 0.55, 0.55]，后 4 步=[0.204, 0.123, 0.069, 0.05]


## 手写核心实现与中间量

以下实现刻意保留关键矩阵、梯度、范数或调度状态，目的是让面试时能解释“它到底改变了哪一个量”。


In [3]:
class LinearRiskHead(nn.Module):  # 定义一个显式 forward 的风险分类头。
    def __init__(self):  # 初始化可训练权重。
        super().__init__()  # 初始化模块父类。
        self.weight = nn.Parameter(torch.randn(3, 2) * 0.12)  # 创建线性分类矩阵。
    def forward(self, batch):  # 写出线性 logits 计算。
        return batch @ self.weight  # 返回风险分类 logits。
def train_schedule(schedule, corrupt_first=False):  # 按给定学习率表执行手写训练。
    torch.manual_seed(23)  # 固定每次训练的初始化以便公平对照。
    model = LinearRiskHead()  # 创建独立分类头。
    losses = []  # 保存每步 loss。
    for step, learning_rate in enumerate(schedule):  # 同时读取 step 和学习率。
        loss = torch.nn.functional.cross_entropy(model(features), labels)  # 计算当前分类损失。
        gradient = torch.autograd.grad(loss, model.weight)[0]  # 获得权重梯度。
        with torch.no_grad():  # 手写参数更新不需构图。
            if corrupt_first and step == 0:  # 模拟没有 warmup 时首步通信符号损坏。
                model.weight += learning_rate * gradient  # 执行错误的上升方向更新。
            else:  # 其他步骤执行正常下降更新。
                model.weight -= learning_rate * gradient  # 按 schedule 更新参数。
        losses.append(float(loss))  # 保存更新前 loss。
    accuracy = float(model(features).argmax(dim=1).eq(labels).float().mean())  # 计算训练结束准确率。
    return losses, accuracy  # 返回完整曲线和准确率。
healthy_losses, healthy_accuracy = train_schedule(healthy_schedule)  # 运行健康 warmup 平台衰减训练。
core_metric = healthy_losses[-1]  # 保存健康 schedule 末步损失。
print(f'健康训练：loss={healthy_losses[0]:.4f}->{healthy_losses[-1]:.4f}，准确率={healthy_accuracy:.2f}')  # 输出核心训练结果。


健康训练：loss=0.6458->0.2455，准确率=1.00


In [4]:
comparison_rows = [('Baseline', float(baseline_metric)), ('核心机制', float(core_metric))]  # 汇总同一指标口径下的可读结果表。
for name, metric in comparison_rows:  # 逐行输出基线与核心方案。
    print(f'{name:<8} | 指标={metric:.6f}')  # 展示结果表而不是只保留变量名。


Baseline | 指标=0.137500
核心机制     | 指标=0.245524


## 结果解读

请把基线和核心输出看成机制证据而非榜单。这里的指标只在同一受控工单集上可比：核心方案展示了 **学习率调度** 的关键状态与更新路径。生产调度应以已消费 token 和恢复后的全局步数为准，记录有效 batch、梯度累积和 skipped step；否则断点续训会把 warmup 重放。

## 失败案例

接下来故意破坏一个必要条件，再用明确的门禁、尺度或统计口径修复它。这样可以避免“代码能跑”却不知道为什么线上会失效。


In [5]:
broken_schedule = [0.55] * 18  # 构造跳过 warmup 和衰减的固定峰值 schedule。
broken_losses, broken_accuracy = train_schedule(broken_schedule, True)  # 在首步注入错误更新以模拟启动事故。
failure_metric = broken_losses[1]  # 记录异常首步之后的 loss。
fix_metric = healthy_losses[1]  # 对齐记录健康 warmup 第二步 loss。
print(f'失败：无 warmup 且首步事故后 loss={failure_metric:.3f}，准确率={broken_accuracy:.2f}；修复：warmup 后同位置 loss={fix_metric:.3f}')  # 展示调度和恢复门限的配合。


失败：无 warmup 且首步事故后 loss=0.724，准确率=1.00；修复：warmup 后同位置 loss=0.627


## 工程取舍、常见坑与延伸追问

**工程取舍：** 生产调度应以已消费 token 和恢复后的全局步数为准，记录有效 batch、梯度累积和 skipped step；否则断点续训会把 warmup 重放。

**常见坑：** 按 dataloader batch 次数而非 token 计调度，或重启训练时把全局 step 归零，会导致重复 warmup 和不可解释的 loss 变化。

**延伸追问：** 为何 batch size warmup 和 learning-rate warmup 可能需要联动？如何在 token 数变化的 packing 方案下保证 schedule 连续？

## 生产差距

本实验只有 6 条脱敏离线工单、CPU 和 FP32，省略了大规模 token packing、数据并行、混合精度、checkpoint、指标告警和灰度回滚。生产实现应替换为真实数据管道与观测系统，并用验证集和线上安全指标决定是否发布。


In [6]:
assert healthy_schedule[0] < healthy_schedule[3]  # 验证 warmup 首步小于峰值。
assert healthy_schedule[-1] < healthy_schedule[9]  # 验证 schedule 在平台后进入衰减。
assert core_metric < healthy_losses[0]  # 验证健康 schedule 在本教学任务上降低了 loss。
assert fix_metric < failure_metric  # 验证 warmup 训练抵御了首步异常。
